# TPE Optimizer (Optuna) — Research Notebook

A reproducible harness for searching strategy hyperparameters using the **Tree-structured Parzen
Estimator** sampler from [Optuna](https://optuna.org/). TPE is a sequential model-based method:
it fits two density estimators (one for "good" trials, one for "bad") and proposes the next trial
where the ratio is highest. Over many trials it concentrates budget in promising regions.

## Why TPE vs. random grid?

- **Supports continuous and log-scale parameters.** The random-grid optimizer requires every
  dimension to be a pre-enumerated tuple; TPE accepts arbitrary `int`/`float`/`categorical`
  ranges, including log-spaced for quantities that span orders of magnitude (entry thresholds,
  stop-loss fractions).
- **Learns from prior trials.** Random search is memoryless; TPE biases future trials toward
  previously-successful regions, which matters more as trial count grows.
- **Same downstream pipeline.** Evaluation, scoring, leaderboard, holdout replay, and chart
  output are identical to the random-grid notebook — only the candidate generation differs.

For the random-grid baseline, see `generic_optimizer_research.ipynb`.

## How to read this notebook

1. **Configuration** — edit data source, market, strategy, `PARAMETER_SPACE`, and budget.
2. **Run optimizer** — Optuna drives trials via ask/tell; each trial is evaluated in an isolated
   subprocess.
3. **Leaderboard** — pandas table of top candidates.
4. **Holdout replay** — chart embedded inline as a persistent iframe.
5. **Caveats** at the bottom — read them.


## Setup

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_candidate / "prediction_market_extensions").is_dir() and (
        _candidate / "backtests"
    ).is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from prediction_market_extensions.backtesting._notebook_support import ensure_notebook_repo_context

repo_root = ensure_notebook_repo_context()

## Configuration

The search space here is a **continuous `PARAMETER_SPACE`** — each key maps to a spec describing
the parameter's type and range:

- `{"type": "int", "low": int, "high": int, "log": bool}` — integer range; set `log=True` for
  log-uniform sampling of wide integer spans.
- `{"type": "float", "low": float, "high": float, "log": bool}` — continuous range; log-uniform
  is strongly recommended for threshold-style parameters that span orders of magnitude.
- `{"type": "categorical", "choices": [...]}` — discrete set; TPE handles these natively.

Unlike the random-grid notebook, you don't enumerate every candidate value — TPE samples
intelligently within the declared bounds. Set `MAX_TRIALS` to your evaluation budget; TPE's
advantage over random grows with budget, so 50–200 is where it really earns its keep.

In [ ]:
from decimal import Decimal

from prediction_market_extensions.backtesting._execution_config import ExecutionModelConfig
from prediction_market_extensions.backtesting._execution_config import StaticLatencyConfig
from prediction_market_extensions.backtesting._experiments import ParameterSearchExperiment
from prediction_market_extensions.backtesting._prediction_market_runner import MarketDataConfig
from prediction_market_extensions.backtesting._replay_specs import QuoteReplay
from prediction_market_extensions.backtesting.data_sources import PMXT, Polymarket, QuoteTick
from prediction_market_extensions.backtesting.optimizers import ParameterSearchConfig
from prediction_market_extensions.backtesting.optimizers import ParameterSearchWindow


NAME = "notebook_tpe_optimizer"

DATA = MarketDataConfig(
    platform=Polymarket,
    data_type=QuoteTick,
    vendor=PMXT,
    sources=(
        "local:/Volumes/LaCie/pmxt_raws",
        "archive:r2.pmxt.dev",
        "relay:209-209-10-83.sslip.io",
    ),
)

BASE_REPLAY = QuoteReplay(
    market_slug="will-ludvig-aberg-win-the-2026-masters-tournament", token_index=0
)

TRAIN_WINDOWS = (
    ParameterSearchWindow(
        name="sample-a-full-window",
        start_time="2026-04-05T00:00:00Z",
        end_time="2026-04-07T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="sample-b-2026-04-06-day",
        start_time="2026-04-06T00:00:00Z",
        end_time="2026-04-06T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="sample-c-2026-04-07-late",
        start_time="2026-04-07T12:00:00Z",
        end_time="2026-04-07T23:59:59Z",
    ),
)

HOLDOUT_WINDOWS = (
    ParameterSearchWindow(
        name="sample-d-close-window",
        start_time="2026-04-07T00:00:00Z",
        end_time="2026-04-07T11:59:59Z",
    ),
)

STRATEGY_SPEC = {
    "strategy_path": "strategies:QuoteTickEMACrossoverStrategy",
    "config_path": "strategies:QuoteTickEMACrossoverConfig",
    "config": {
        "trade_size": Decimal("5"),
        "fast_period": "__SEARCH__:fast_period",
        "slow_period": "__SEARCH__:slow_period",
        "entry_buffer": "__SEARCH__:entry_buffer",
        "take_profit": "__SEARCH__:take_profit",
        "stop_loss": "__SEARCH__:stop_loss",
    },
}

# Continuous / log-scale space — TPE samples within these bounds.
PARAMETER_SPACE = {
    "fast_period": {"type": "int", "low": 16, "high": 128},
    "slow_period": {"type": "int", "low": 96, "high": 512},
    "entry_buffer": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
    "take_profit": {"type": "float", "low": 2e-3, "high": 3e-2, "log": True},
    "stop_loss": {"type": "float", "low": 2e-3, "high": 3e-2, "log": True},
}

EXECUTION = ExecutionModelConfig(
    queue_position=True,
    latency_model=StaticLatencyConfig(
        base_latency_ms=75.0,
        insert_latency_ms=10.0,
        update_latency_ms=5.0,
        cancel_latency_ms=5.0,
    ),
)

MAX_TRIALS = 50
HOLDOUT_TOP_K = 5

PARAMETER_SEARCH = ParameterSearchConfig(
    name=NAME,
    data=DATA,
    base_replay=BASE_REPLAY,
    strategy_spec=STRATEGY_SPEC,
    parameter_space=PARAMETER_SPACE,
    train_windows=TRAIN_WINDOWS,
    holdout_windows=HOLDOUT_WINDOWS,
    max_trials=MAX_TRIALS,
    random_seed=7,
    holdout_top_k=HOLDOUT_TOP_K,
    initial_cash=100.0,
    probability_window=256,
    min_quotes=500,
    min_price_range=0.005,
    min_fills_per_window=1,
    execution=EXECUTION,
    emit_html=False,
    chart_output_path="output",
    sampler="tpe",
)

EXPERIMENT = ParameterSearchExperiment(
    name=NAME,
    description="EMA TPE search on PMXT L2 data",
    parameter_search=PARAMETER_SEARCH,
)

print(
    f"Configured TPE search: {MAX_TRIALS} trials over "
    f"{len(TRAIN_WINDOWS)} train window(s), {len(HOLDOUT_WINDOWS)} holdout window(s)."
)

## Run optimizer

Optuna drives trials using its ask/tell API, so each trial is evaluated inside an isolated
subprocess (same harness as the random-grid notebook). Expect this to take significantly longer
than a random-grid run at the same budget — that's the cost of sequential model updates, and
it's usually worth it.

In [ ]:
from pprint import pprint

from prediction_market_extensions.backtesting._experiments import run_experiment_async
from prediction_market_extensions.backtesting._notebook_support import suppress_notebook_cell_output

with suppress_notebook_cell_output():
    summary = await run_experiment_async(EXPERIMENT)

print("Selected params:")
pprint(dict(summary.selected_params))
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

## Leaderboard

Top candidates as ranked by the risk-adjusted score. With a continuous search space, TPE often
finds near-optimal regions that wouldn't exist on any discrete grid — watch for parameter values
that don't look like round numbers.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

_leaderboard_df = pd.read_csv(summary.leaderboard_csv_path)
_display_cols = [
    c for c in (
        "trial_id",
        "train_median_score",
        "holdout_median_score",
        "train_median_pnl",
        "holdout_median_pnl",
        "train_median_drawdown",
        "train_median_fills",
        "params_json",
    )
    if c in _leaderboard_df.columns
]
_top = _leaderboard_df[_display_cols].head(10).copy()

_fmt = {
    "train_median_score": "{:.4f}".format,
    "holdout_median_score": "{:.4f}".format,
    "train_median_pnl": "{:.4f}".format,
    "holdout_median_pnl": "{:.4f}".format,
    "train_median_drawdown": "{:.4f}".format,
    "train_median_fills": "{:.1f}".format,
}
_fmt = {k: v for k, v in _fmt.items() if k in _top.columns}

_styled = (
    _top.style
        .format(_fmt, na_rep="-")
        .hide(axis="index")
        .set_caption(f"Top candidates — {EXPERIMENT.name} (TPE)")
)
display(_styled)

display(Markdown(
    "### Selected parameters\n\n"
    + "\n".join(f"- **{k}**: `{v}`" for k, v in dict(summary.selected_params).items())
))

## Holdout replay & chart

Replays the TPE-selected parameters on the first holdout window with full chart output. Embedded
inline as a persistent iframe.

In [ ]:
from prediction_market_extensions.backtesting._notebook_support import (
    display_html_artifacts,
    find_updated_html_artifacts,
    snapshot_html_artifacts,
    suppress_notebook_cell_output,
)
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)

selected_window = HOLDOUT_WINDOWS[0] if HOLDOUT_WINDOWS else TRAIN_WINDOWS[-1]

output_root = repo_root / "output"
html_snapshot = snapshot_html_artifacts(output_root)

backtest = build_parameter_search_window_backtest(
    config=PARAMETER_SEARCH,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{NAME}:{selected_window.name}:selected-params",
    emit_html=True,
    chart_output_path=str(output_root),
    return_summary_series=False,
)

with suppress_notebook_cell_output():
    results = await backtest.run_async()

updated_html = find_updated_html_artifacts(output_root, html_snapshot)
print(f"Replayed {len(results)} market(s) on window '{selected_window.name}'.")
display_html_artifacts(updated_html, repo_root=repo_root)

## Caveats & next steps

- **Overfitting is the default outcome.** TPE concentrates budget in regions that score well on
  the training windows — that makes holdout scrutiny even more important than for random search.
- **Trial budget matters.** TPE's early trials are essentially random (cold-start); its advantage
  kicks in after ~10–20 trials. Budgets under ~20 often tie with random. Prefer `MAX_TRIALS >= 50`.
- **Reproducibility** — seeded via `random_seed`, but subprocess nondeterminism in the backtest
  engine can introduce small score jitter. Run the same seed twice if you need to confirm.
- **Continuous ≠ always better.** Only use a continuous range when the strategy is actually
  smooth in that dimension. Integer periods (fast/slow) and log-scale TP/SL typically are; some
  parameters are inherently discrete (mode flags, window counts).
- **Parallelism is deferred.** TPE benefits from seeing prior results before proposing the next
  trial, so trials run sequentially for now. Parallel ask/tell is a follow-on.
- Artifacts land in `output/`: `*_leaderboard.csv`, `*_summary.json`, and the chart HTML.
